# 🎬 Global YouTube Statistics — Data Science Analysis
> **Portfolio Project** · Dataset: Top 995 YouTube Channels (2023) · Source: Kaggle

## Objective
Explore what separates the biggest YouTube channels from the rest:
- Which **categories** dominate in views vs. earnings?
- Which **countries** produce the most top creators?
- Is there a link between **subscribers**, **views** and **revenue**?
- How has YouTube **grown** over time?

---


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
COLORS = px.colors.qualitative.Plotly


In [ ]:
df = pd.read_csv('data/Global_YouTube_Statistics.csv', encoding='ISO-8859-1')
# Fix BOM on first column
df.columns = [c.lstrip('\ufeff') for c in df.columns]
print(f"Shape: {df.shape}")
df.head(3)


## 2. Exploratory Data Analysis

In [ ]:
# Missing values summary
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
print(missing.to_string())


In [ ]:
# Basic stats on key numeric columns
df[['subscribers', 'video views', 'uploads',
    'lowest_monthly_earnings', 'highest_yearly_earnings']].describe().round(0)


### 2.1 Data Cleaning

Key decisions:
- **Duplicates**: Removed based on `Title`
- **1970 outlier**: One channel had `created_year = 1970` — clearly erroneous, removed
- **Missing country/category**: Kept rows for non-geographic analyses, dropped for map/country charts
- **Encoding**: ISO-8859-1 to handle special characters in channel names


In [ ]:
df = df.drop_duplicates(subset='Title')
df = df[df['created_year'] != 1970]

df_geo   = df[df['Country'] != 'nan'].dropna(subset=['Country', 'Latitude', 'Longitude'])
df_cat   = df.dropna(subset=['category'])

print(f"Clean dataset: {len(df)} channels | {df['Country'].nunique()} countries")


## 3. Which Category Should You Target?

### 3.1 Views vs. Earnings — The category trade-off

Not all popular categories are equally lucrative. Music has the most views but advertisers pay
differently per niche. Let's compare **total views** and **average yearly earnings** by category.


In [ ]:
cat_views   = df_cat.groupby('category')['video views'].sum().sort_values(ascending=False)
cat_earn    = df_cat.groupby('category')['highest_yearly_earnings'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Total views ---
ax = axes[0]
bars = ax.barh(cat_views.index[::-1], cat_views.values[::-1] / 1e12, color='#4B8BBE')
ax.set_xlabel('Total Views (Trillions)', fontsize=11)
ax.set_title('Total Views by Category', fontsize=13, fontweight='bold', pad=15)
for bar, val in zip(bars, cat_views.values[::-1] / 1e12):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}T', va='center', fontsize=9)

# --- Right: Avg earnings ---
ax = axes[1]
bars = ax.barh(cat_earn.index[::-1], cat_earn.values[::-1] / 1e6, color='#E07B54')
ax.set_xlabel('Avg Highest Yearly Earnings ($M)', fontsize=11)
ax.set_title('Avg Yearly Earnings by Category', fontsize=13, fontweight='bold', pad=15)
for bar, val in zip(bars, cat_earn.values[::-1] / 1e6):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f'${val:.1f}M', va='center', fontsize=9)

fig.suptitle('Views ≠ Money: Music Dominates Views, Shows Leads in Earnings',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visuals/category_views_vs_earnings.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Insight: Music leads views 3x over the next category, but 'Shows' channels earn more on average.")


## 4. Geographic Distribution — Where Are Top Creators?

### 4.1 World Map — Channel count & total subscribers by country


In [ ]:
country_stats = df_geo.groupby('Country').agg(
    channel_count=('Youtuber', 'count'),
    total_subscribers=('subscribers', 'sum'),
    avg_subscribers=('subscribers', 'mean')
).reset_index()
country_stats['total_subscribers_M'] = (country_stats['total_subscribers'] / 1e6).round(1)

fig = px.choropleth(
    country_stats,
    locations='Country',
    locationmode='country names',
    color='channel_count',
    hover_name='Country',
    hover_data={'channel_count': True, 'total_subscribers_M': ':.0f'},
    color_continuous_scale='Blues',
    title='<b>Number of Top-1000 YouTube Channels per Country</b>',
    labels={'channel_count': 'Channel Count', 'total_subscribers_M': 'Total Subscribers (M)'}
)
fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True, projection_type='equirectangular'),
    coloraxis_colorbar=dict(title='Channels'),
    title_font_size=16,
    margin=dict(l=0, r=0, t=50, b=0)
)
fig.write_html('visuals/map_channels_by_country.html')
fig.show()
print("✅ Insight: The US alone accounts for 31% of top-1000 channels. India is second at 17%.")


## 5. Subscribers, Views & Earnings — How correlated are they?

### 5.1 Correlation Heatmap


In [ ]:
num_cols = ['subscribers', 'video views', 'uploads',
            'video_views_for_the_last_30_days',
            'lowest_monthly_earnings', 'highest_yearly_earnings',
            'subscribers_for_last_30_days']
corr_matrix = df[num_cols].dropna().corr()

# Rename for readability
labels = {
    'subscribers': 'Subscribers',
    'video views': 'Total Views',
    'uploads': 'Uploads',
    'video_views_for_the_last_30_days': 'Views (30d)',
    'lowest_monthly_earnings': 'Min Monthly $',
    'highest_yearly_earnings': 'Max Yearly $',
    'subscribers_for_last_30_days': 'New Subs (30d)'
}
corr_matrix = corr_matrix.rename(index=labels, columns=labels)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Between Key Channel Metrics',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('visuals/correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Insight: Total views correlates strongly with earnings (0.55) — more than subscriber count alone (0.39).")


### 5.2 Subscribers vs. Earnings — Scatter by Category

In [ ]:
scatter_df = df_cat.dropna(subset=['highest_yearly_earnings']).copy()
scatter_df['earnings_M'] = scatter_df['highest_yearly_earnings'] / 1e6
scatter_df['subscribers_M'] = scatter_df['subscribers'] / 1e6
scatter_df['views_B'] = scatter_df['video views'] / 1e9

fig = px.scatter(
    scatter_df,
    x='subscribers_M',
    y='earnings_M',
    color='category',
    size='views_B',
    hover_name='Youtuber',
    hover_data={'subscribers_M': ':.1f', 'earnings_M': ':.1f', 'category': True},
    title='<b>Subscribers vs. Yearly Earnings</b> (bubble size = total views)',
    labels={
        'subscribers_M': 'Subscribers (Millions)',
        'earnings_M': 'Highest Yearly Earnings ($M)',
        'category': 'Category'
    },
    template='plotly_white',
    opacity=0.75
)
fig.update_layout(title_font_size=15, legend_title_text='Category')
fig.write_html('visuals/scatter_subscribers_earnings.html')
fig.show()


## 6. YouTube Channel Creation Over Time

When did the YouTube creator boom happen?


In [ ]:
timeline = df.dropna(subset=['created_year']).copy()
timeline['created_year'] = timeline['created_year'].astype(int)
year_count = timeline.groupby('created_year').size().reset_index(name='new_channels')
year_count = year_count[(year_count['created_year'] >= 2005) & (year_count['created_year'] <= 2022)]

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(year_count['created_year'], year_count['new_channels'],
                alpha=0.25, color='#4B8BBE')
ax.plot(year_count['created_year'], year_count['new_channels'],
        color='#4B8BBE', linewidth=2.5, marker='o', markersize=6)

# Annotate key events
events = {2006: 'Google\nacquires\nYouTube', 2012: 'YouTube\nPartner\nProgram\nexpands',
           2016: 'Peak new\nchannels\ncreated'}
for year, label in events.items():
    y_val = year_count[year_count['created_year'] == year]['new_channels'].values
    if len(y_val):
        ax.annotate(label, xy=(year, y_val[0]), xytext=(year, y_val[0]+15),
                    ha='center', fontsize=9, color='#555',
                    arrowprops=dict(arrowstyle='-', color='#999'))

ax.set_title('When Were Today\'s Top Channels Created?\nThe Creator Boom (2005–2022)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Year Created', fontsize=11)
ax.set_ylabel('Number of Top-1000 Channels Created', fontsize=11)
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.tight_layout()
plt.savefig('visuals/channel_creation_timeline.png', bbox_inches='tight', dpi=150)
plt.show()


## 7. Distribution of Subscribers by Category — Violin Plot

A violin plot reveals not just averages but the *shape* of the distribution — are channels clustered near the bottom or spread out?


In [ ]:
violin_df = df_cat.copy()
violin_df['subscribers_M'] = violin_df['subscribers'] / 1e6
# Keep top categories by count for readability
top_cats = violin_df['category'].value_counts().head(10).index
violin_df = violin_df[violin_df['category'].isin(top_cats)]

fig, ax = plt.subplots(figsize=(14, 6))
order = violin_df.groupby('category')['subscribers_M'].median().sort_values(ascending=False).index

sns.violinplot(data=violin_df, x='category', y='subscribers_M',
               order=order, palette='muted', inner='quartile', ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_title('Subscriber Distribution by Category\n(Top 10 Categories by Channel Count)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Category', fontsize=11)
ax.set_ylabel('Subscribers (Millions)', fontsize=11)
plt.tight_layout()
plt.savefig('visuals/violin_subscribers_by_category.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Insight: Music and Entertainment have wide distributions — a few megachannels skew the average significantly.")


## 8. Key Takeaways

| Question | Finding |
|---|---|
| 🎵 **Best category for views?** | **Music** — 3.1 trillion total views, 2× more than Entertainment |
| 💰 **Best category for earnings?** | **Shows** & **Autos** — higher CPM, fewer channels competing |
| 🌍 **Where are creators?** | **USA (31%)** and **India (17%)** dominate — but emerging markets are growing |
| 📈 **Subscribers vs earnings?** | **Views matter more than subs** for revenue (corr: 0.55 vs 0.39) |
| 🕐 **When to have started?** | Peak channel creation was **2012–2016** — the platform matures fast |

---

*Data source: [Global YouTube Statistics 2023 — Kaggle](https://www.kaggle.com/datasets/nelgiriyewithana/global-youtube-statistics-2023)*
